# PathSnap dataset exploration
This notebook covers the `train` and `student_test` partitions. Each trip directory holds headerless, whitespace-separated `.track`, `.nodes`, `.arcs` and `.route` files. The first two use longitude and latitude. The second two store integer IDs that refer to zero-based row positions in their associated files.
Set `PATHSNAP_DATA` to the directory containing both partitions before opening the notebook. The optional Folium outputs stay under `data/processed/` because they contain GPS geometry.


In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
from IPython.display import Image, display

root = Path.cwd()
if not (root / 'config.py').is_file():
    root = root.parent
output = root / 'results' / 'eda'
maps = root / 'data' / 'processed' / 'eda_maps'
subprocess.run([sys.executable, str(root / 'notebooks' / 'dataset_exploration.py'),
                '--out', str(output), '--maps-dir', str(maps)], check=True, cwd=root)


## Partition and file inventory
File summaries report bytes by partition and extension. Per-trip counts, spatial extent, and GPS and road-network checks come from the raw files. The reference coverage fields come from the processed per-trip tables when present.


In [ ]:
summary = pd.read_csv(output / 'partition_summary.csv')
files = pd.read_csv(output / 'file_summary.csv')
trips = pd.read_csv(output / 'trips.csv', dtype={'trip_id': str})
display(summary[['partition', 'trips', 'n_obs_total', 'n_obs_min', 'n_obs_median', 'n_obs_max']])
display(files)


## Spatial extent and trajectories
The histograms use the percentage of GPS observations within each partition. The example traces use a trip-local UTM projection and relative kilometres. Green marks the start and orange marks the end.


In [ ]:
for filename in ('gps_distributions.png', 'trajectory_examples.png'):
    display(Image(filename=str(output / filename)))
print('Local Folium HTML files')
for path in sorted(maps.glob('*.html')):
    print(path.name)


## Connectivity and data quality
Weak components ignore arc direction when checking connectivity. An isolated node has no incident arc. A disconnected reference step is a pair of consecutive valid arc IDs whose junctions do not join directly.


In [ ]:
cols = ['partition', 'trip_id', 'n_nodes', 'n_arcs', 'weak_components',
        'largest_component_fraction', 'isolated_nodes']
display(trips[cols].sort_values('weak_components', ascending=False).head(10))
display(trips[['partition', 'trip_id', 'max_gap_s', 'gaps_gt_10s',
               'gps_rows_with_nonfinite', 'disconnected_route_steps']]
        .sort_values('max_gap_s', ascending=False).head(10))
if 'pct_ref_valid' in trips:
    display(trips[['partition', 'trip_id', 'pct_ref_valid', 'ref_dist_median_m']]
            .sort_values('pct_ref_valid').head(8))
